# Week 6 Practice: CNN Foundations

CSCI1409 Deep Learning · based on Lecture 5

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Do not use AI tools for any task (ChatGPT, Claude, Gemini, Copilot, Google AI Overviews).
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Task 1: Convolution

A convolution slides a small kernel over the input. At each position it multiplies the matching entries, sums them and adds a bias. The same kernel is used at every position.

We use the example from the lecture:

$$X = \begin{bmatrix} 1 & 2 & 0 & 1 \\ 0 & 1 & 3 & 2 \\ 2 & 2 & 1 & 0 \\ 1 & 0 & 2 & 3 \end{bmatrix}, \qquad K = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}$$

### 1a) One convolution by hand

1. Compute all four entries of $Y$ on paper (stride 1, padding 0, bias 0). Write the calculation for each entry.
2. Check your answer with `F.conv2d`. It expects the input as `[B, C, H, W]` and the kernel as `[C_out, C_in, k, k]`, so use `.view(1, 1, 4, 4)` and `.view(1, 1, 3, 3)`.
3. What would $Y$ be with bias $b = 1$?

In [ ]:
X = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 2.],
                  [2., 2., 1., 0.],
                  [1., 0., 2., 3.]])

K = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])

# Your code here
conv1 = nn.Conv2d(1, 1, 3, 1, 0, bias=False)

with torch.no_grad():
    conv1.weight[0, 0] = K

X = X.unsqueeze(0).unsqueeze(0)

Y = conv1(X)

loss = Y.sum()

loss.backward()

print("Output:", Y)
print("Loss:", loss.item())
print("Gradients:", conv1.weight.grad)


X_u = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 2.],
                  [2., 2., 1., 0.],
                  [1., 0., 2., 3.]])

K_u = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])

X = X.view(1, 1, 4, 4)
K = K.view(1, 1, 3, 3)

# Convolution without bias
Y = F.conv2d(X, K, stride=1, padding=0)

print("Without bias:")
print(Y)

# Convolution with bias = 1
bias = torch.tensor([1.0])

Y_bias = F.conv2d(X, K, bias=bias, stride=1, padding=2)

print("With bias = 1:")
print(Y_bias)

Output: tensor([[[[-1.,  2.],
          [-3., -2.]]]], grad_fn=<ConvolutionBackward0>)
Loss: -4.0
Gradients: tensor([[[[4., 6., 6.],
          [5., 7., 6.],
          [5., 5., 6.]]]])
Without bias:
tensor([[[[-1.,  2.],
          [-3., -2.]]]])
With bias = 1:
tensor([[[[ 0.,  3.],
          [-2., -1.]]]])


**Your answer:**

### 1b) Output size and parameters

From the lecture:

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1, \qquad \text{params} = C_{out}\,(C_{in}\,k^2 + 1)$$

| Case | Input | Filters | $k$ | $p$ | $s$ |
|---|---|---|---|---|---|
| A | 1×28×28 | 8 | 3 | 0 | 1 |
| B | 1×28×28 | 8 | 3 | 1 | 1 |
| C | 1×28×28 | 8 | 3 | 1 | 2 |
| D | 3×32×32 | 16 | 5 | 2 | 1 |

1. Use the formulas to compute the output shape and the number of parameters for each case.
2. Check with `nn.Conv2d`. Pass `torch.randn(1, C_in, H, H)` through the layer, print the output shape, and count the parameters with `sum(p.numel() for p in layer.parameters())`.
3. A, B and C have the same number of parameters. Why?

In [44]:
# Your code here
cases = [
    ("A", 1, 28, 8, 3, 0, 1),
    ("B", 1, 28, 8, 3, 1, 1),
    ("C", 1, 28, 8, 3, 1, 2),
    ("D", 3, 32, 16, 5, 2, 1)
]

for name, con_i, im_s, k_n, k_s, p, s in cases:
    H_out = ((im_s + 2*p - k_s) // s) + 1
    params = k_n * (con_i * k_s**2 + 1)
    print(f"Case {name}:")
    print(f"Output Shape: ({k_n}, {H_out}, {H_out})")
    print(f"Parameters: {params}")
    print()
    
    conv_l = nn.Conv2d(con_i, k_n, k_s, stride=s, padding=p)
    X = torch.randn(1, con_i, im_s, im_s)
    
    Y = conv_l(X)
    actual_params = sum(param.numel() for param in conv_l.parameters())
    print(f"Case {name}:")
    print(f"PyTorch Output Shape: {Y.shape}")
    print(f"PyTorch Parameters: {actual_params}")
    print()

Case A:
Output Shape: (8, 26, 26)
Parameters: 80

Case A:
PyTorch Output Shape: torch.Size([1, 8, 26, 26])
PyTorch Parameters: 80

Case B:
Output Shape: (8, 28, 28)
Parameters: 80

Case B:
PyTorch Output Shape: torch.Size([1, 8, 28, 28])
PyTorch Parameters: 80

Case C:
Output Shape: (8, 14, 14)
Parameters: 80

Case C:
PyTorch Output Shape: torch.Size([1, 8, 14, 14])
PyTorch Parameters: 80

Case D:
Output Shape: (16, 32, 32)
Parameters: 1216

Case D:
PyTorch Output Shape: torch.Size([1, 16, 32, 32])
PyTorch Parameters: 1216



**Your answer:**

## Task 2: ReLU, pooling and the full CNN

### 2a) ReLU and max pooling

$Y$ is your result from 1a. $P$ is the matrix from the max pooling slide:

$$P = \begin{bmatrix} 1 & 3 & 2 & 0 \\ 4 & 2 & 1 & 5 \\ 0 & 1 & 3 & 2 \\ 2 & 0 & 1 & 4 \end{bmatrix}$$

1. Predict $\text{ReLU}(Y)$ and 2×2 max pooling with stride 2 of $P$ by hand.
2. Check with `nn.ReLU()` and `nn.MaxPool2d(2)`.
3. A batch of shape B×8×28×28 goes through `nn.MaxPool2d(2)`. What is the output shape? Why does the number of channels not change?

In [51]:
P = torch.tensor([[1., 3., 2., 0.],
                  [4., 2., 1., 5.],
                  [0., 1., 3., 2.],
                  [2., 0., 1., 4.]])

# Your code here
Y = torch.tensor([
    [-1.,  2.],
    [-3., -2.]
])

relu = nn.ReLU()
print(relu(Y))

pool = nn.MaxPool2d(kernel_size=2, stride=2)
P = P.unsqueeze(0).unsqueeze(0)
print(pool(P))
print(pool(P).shape)

tensor([[0., 2.],
        [0., 0.]])
tensor([[[[4., 5.],
          [2., 4.]]]])
torch.Size([1, 1, 2, 2])


**Your answer:**

### 2b) The lecture CNN

1. Build the CNN from Lecture 5 (slide "The CNN in PyTorch") with `nn.Sequential`. Pass `x = torch.randn(4, 1, 28, 28)` through it one layer at a time and print the shape after each layer. Hint: `for layer in model: x = layer(x)`.
2. Count the parameters of the whole model and of each layer. What is the flattened feature count? Which layer has the most parameters?
3. An MLP for the same task (Flatten, Linear(784, 128), ReLU, Linear(128, 10)) has 101,770 parameters. Why does the CNN need so many fewer?

In [53]:
# Your code here
model = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2), 
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10)
)
x = torch.randn(4, 1, 28, 28)
for layer in model: 
    x = layer(x)
    print(layer.__class__.__name__, "-", x.shape)
    
total_params = 0

for layer in model:
    params = sum(p.numel() for p in layer.parameters())

    print(layer.__class__.__name__, ":", params)

    total_params += params

print("Total:", total_params)
    

Conv2d - torch.Size([4, 8, 28, 28])
ReLU - torch.Size([4, 8, 28, 28])
MaxPool2d - torch.Size([4, 8, 14, 14])
Conv2d - torch.Size([4, 16, 14, 14])
ReLU - torch.Size([4, 16, 14, 14])
MaxPool2d - torch.Size([4, 16, 7, 7])
Flatten - torch.Size([4, 784])
Linear - torch.Size([4, 10])
Conv2d : 80
ReLU : 0
MaxPool2d : 0
Conv2d : 1168
ReLU : 0
MaxPool2d : 0
Flatten : 0
Linear : 7850
Total: 9098


**Your answer:**

## Stretch (optional): convolution with loops

Write `conv2d_manual(X, K)` with two `for` loops (stride 1, padding 0, no bias). At output position $(i, j)$ take the patch `X[i:i+k, j:j+k]`, multiply it with K element by element and sum. Check that you get the same $Y$ as in 1a.

In [47]:
# Your code here